# Logistic Regression Training (PySpark ML)

Train logistic regression models for network intrusion detection using PySpark ML to handle large datasets (10GB+).

## Strategies:
1. **Balanced Data**: Train on oversampled balanced dataset
2. **Class Weighting**: Use class weights to handle imbalance

In [ ]:
import importlib
import os
import sys
sys.path.append('..')

from pathlib import Path

# Reload to pick up training_utils changes without kernel restart
import notebooks.training_utils
importlib.reload(notebooks.training_utils)
from notebooks.training_utils import (
    load_training_data_pyspark,
    train_and_evaluate_pyspark,
    save_models_pyspark,
    print_summary_pyspark
)

os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17'
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.ml.classification import LogisticRegression

# Stop any existing Spark session
try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
        print("Stopped existing Spark session")
except:
    pass

JVM_FLAGS = (
"-XX:+UseG1GC "
"-XX:MaxGCPauseMillis=500 "
"-XX:InitiatingHeapOccupancyPercent=35 "
"-XX:+UseStringDeduplication"
)

# Create Spark session optimised for M4 Mac (16 GB RAM, 10-core)
# reducing OS memory pressure and the risk of swapping during training.
spark = SparkSession.builder \
    .appName("LogisticRegressionTraining") \
    .master("local[4]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.driver.extraJavaOptions", JVM_FLAGS) \
    .config("spark.sql.shuffle.partitions", "20") \
    .config("spark.default.parallelism", "8") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "512m") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "false") \
    .config("spark.sql.inMemoryColumnarStorage.batchSize", "2000") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .getOrCreate()

print(f"✓ Spark initialized")
print(f"Spark Version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")


Stopped existing Spark session
✓ Spark initialized
Spark Version: 3.5.6
Spark UI: http://ip-192-168-1-142.eu-west-1.compute.internal:4041


26/03/06 15:09:49 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## 1. Load Data

In [ ]:
# Load data (verbose=False skips row counts for faster loading; restart kernel if TypeError)
train_orig_vec, train_balanced_vec, test_vec, feature_cols, project_root = load_training_data_pyspark(spark, verbose=False)


Loading training and test data from Parquet with PySpark...
✓ Data loaded (93 features, counts deferred for speed)
✓ Assembled feature vectors (lazy — will be cached partition-by-partition during training)


In [7]:
# Data is already prepared by the utility function
print(f"✓ Ready for training with {len(feature_cols)} features")

✓ Ready for training with 93 features


## 2. Train Models

In [ ]:
### Strategy 1: Balanced Data

In [7]:
# Strategy 1: Balanced Data
model_params = {
    'featuresCol': 'features',
    'labelCol': 'label',
    'maxIter': 50,           # Reduced from 100 — LR typically converges in 20-40 iters
    'tol': 1e-3,             # Loosened from 1e-4 — stops earlier with negligible accuracy loss
    'regParam': 0.01,
    'elasticNetParam': 0.0,  # L2 regularization
    'family': 'binomial',
    'aggregationDepth': 4,   # Speeds up treeAggregate for many features/partitions
}

model_balanced, metrics_balanced, _ = train_and_evaluate_pyspark(
    LogisticRegression,
    model_params,
    train_balanced_vec,
    test_vec,
    "Logistic Regression - Balanced Data Strategy",
    use_class_weights=False
)


TRAINING: Logistic Regression - Balanced Data Strategy
  Checkpointing training data to /tmp/spark_checkpoints …


26/03/06 15:10:45 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/03/06 15:13:39 WARN NettyRpcEnv: Ignored message: HeartbeatResponse(false) 8]
26/03/06 15:13:33 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.rpc.RpcTimeoutException: Futures timed out after [10000 milliseconds]. This timeout is controlled by spark.executor.heartbeatInterval
	at org.apache.spark.rpc.RpcTimeout.org$apache$spark$rpc$RpcTimeout$$createRpcTimeoutException(RpcTimeout.scala:47)
	at org.apache.spark.rpc.RpcTimeout$$anonfun$addMessageIfTimeout$1.applyOrElse(RpcTimeout.scala:62)
	at org.apache.spark.rpc.RpcTimeout$$anonfun$addMessageIfTimeout$1.applyOrElse(RpcTimeout.scala:58)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:76)
	at org.apache.spark.rpc.RpcEndp

ConnectionRefusedError: [Errno 61] Connection refused

ConnectionRefusedError: [Errno 61] Connection refused

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/Users/matthewweaver/Repositories/nidstream/.venv/lib/python3.11/site-packages/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/matthewweaver/Repositories/nidstream/.venv/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/matthewweaver/Repositories/nidstream/.venv/lib/python3.11/site-packages/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


In [ ]:
# Strategy 2: Class Weighting
model_weighted, metrics_weighted, _ = train_and_evaluate_pyspark(
    LogisticRegression,
    model_params,
    train_orig_vec,
    test_vec,
    "Logistic Regression - Class Weight Strategy",
    use_class_weights=True
)

## 3. Save Models

In [ ]:
# Save models and metrics
save_models_pyspark(model_balanced, model_weighted, metrics_balanced, metrics_weighted, 'lr', project_root)

## 4. Summary

In [ ]:
# Print comparison summary
print_summary_pyspark(metrics_balanced, metrics_weighted, "Logistic Regression")

In [ ]:
# Stop Spark session when done
# spark.stop()